%md
# Preprocesamiento, validación y análisis de ventas retail con Apache Spark en Databricks

**MCDI502 · Gestión de Datos y Tecnologías Big Data** · Sumativa 4 (Taller III) · Universidad Andrés Bello

**Grupo 6:** Fernanda Ovalle Román, Sebastián Cajales Cid, César Lorca Bacián, Jorge Álvarez Ossandón

---

## Objetivo

Construir un flujo completo de ingeniería de datos en un entorno distribuido: integrar las ventas de tres sucursales de una cadena de retail, depurar sus problemas de calidad, validar el resultado mediante reglas e indicadores objetivos y analizar el conjunto limpio con Spark SQL, de modo que sirva como base confiable para la toma de decisiones.

## Caso de estudio

Se dispone de las ventas del cuarto trimestre de 2024 de las sucursales de **Viña del Mar**, **Santiago** y **Concepción**, cada una en un archivo CSV independiente con la misma estructura de nueve columnas:

| Columna | Descripción |
|---|---|
| `id_transaccion` | Identificador de la venta |
| `fecha` | Fecha de la transacción |
| `id_producto` | Identificador del producto |
| `producto_nombre` | Nombre del producto |
| `categoria` | Categoría del producto |
| `cantidad` | Unidades vendidas |
| `precio_unitario` | Precio por unidad (CLP) |
| `id_cliente` | Identificador del cliente |
| `sucursal` | Sucursal de origen |

Los archivos presentan problemas de calidad que deben resolverse antes del análisis: registros duplicados, categorías faltantes, fechas en múltiples formatos, precios con símbolos monetarios, nombres de productos inconsistentes y cantidades no válidas.

## Estructura del notebook

| Parte | Contenido | Indicador |
|---|---|---|
| 0 | Configuración del entorno | — |
| 1 | Carga e integración de datos | ID3.1 |
| 2 | Limpieza y depuración | ID4.2 |
| 3 | Transformación y análisis con DataFrames | ID4.1 |
| 4 | Validación y control de calidad | ID4.3 |
| 5 | Análisis con Spark SQL | ID3.3 |
| 6 | Conclusiones | — |

## Criterio de trazabilidad

Cada operación que puede alterar el número de registros (unión, eliminación de duplicados y filtrado) se acompaña de un control explícito del conteo de filas, registrado en una bitácora que se presenta en la Parte 4. Así, toda diferencia entre el volumen inicial y el final queda cuantificada y justificada.

%md
## Parte 0. Configuración del entorno

En Databricks la sesión de Spark ya está disponible en la variable `spark`, por lo que no es necesario crearla ni configurar Java. En esta celda se importan los módulos de PySpark que se usarán a lo largo del notebook, se define la ruta del *volume* de Unity Catalog donde se cargaron los archivos y se crea la **bitácora de trazabilidad**.

**Decisión técnica (almacenamiento).** El enunciado indica cargar los archivos en el Databricks File System (DBFS). En Databricks Free Edition el DBFS raíz está deshabilitado y su reemplazo es un *volume* de Unity Catalog, que cumple la misma función: almacenamiento de archivos accesible desde Spark mediante una ruta. Los tres CSV se cargaron en `/Volumes/workspace/default/ventas`.

**Bitácora de trazabilidad.** La función `registrar()` cuenta las filas de un DataFrame y guarda el par (paso, filas) en una lista. Se invoca después de cada operación que puede modificar el número de registros, lo que permite reconstruir en la Parte 4 el recorrido completo de los datos.

In [0]:
# Módulos de PySpark utilizados en el notebook
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Ruta del volume de Unity Catalog donde se cargaron los tres archivos fuente
RUTA_DATOS = "/Volumes/workspace/default/ventas"

# Bitácora de trazabilidad: lista de tuplas (paso, número de filas)
bitacora = []

def registrar(paso, df):
    """Cuenta las filas de df, las agrega a la bitácora y devuelve el conteo."""
    n = df.count()
    bitacora.append((paso, n))
    print(f"{paso}: {n} filas")
    return n

# Verificación del entorno: versión de Spark y archivos disponibles en el volume
print(f"Versión de Apache Spark: {spark.version}")
display(dbutils.fs.ls(RUTA_DATOS))

Versión de Apache Spark: 4.2.0


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/ventas/ventas_concepcion.csv,ventas_concepcion.csv,7095,1794669111000
dbfs:/Volumes/workspace/default/ventas/ventas_santiago.csv,ventas_santiago.csv,9940,1794669111000
dbfs:/Volumes/workspace/default/ventas/ventas_vina_del_mar.csv,ventas_vina_del_mar.csv,8946,1794669111000


%md
**Resultado.** La sesión de Spark está operativa y el *volume* contiene los tres archivos fuente: `ventas_concepcion.csv` (7.095 bytes), `ventas_santiago.csv` (9.940 bytes) y `ventas_vina_del_mar.csv` (8.946 bytes). La ruta con prefijo `dbfs:/Volumes/...` confirma que el *volume* se accede con la misma interfaz de archivos que el DBFS, por lo que el reemplazo no altera el flujo de trabajo. El entorno queda listo para la carga de datos.

%md
## Parte 1. Carga e integración de datos (ID3.1)

### 1.1 Carga de los tres archivos en DataFrames independientes

Cada archivo se carga en su propio DataFrame (`df_vina`, `df_santiago`, `df_concepcion`) mediante una función común, lo que garantiza que los tres se lean con exactamente las mismas opciones:

| Opción | Valor | Justificación |
|---|---|---|
| `header` | `True` | La primera línea contiene los nombres de las columnas |
| `inferSchema` | `False` | Todas las columnas se leen como texto (`string`) |
| `encoding` | `UTF-8` | Preserva las tildes y la "ñ" (por ejemplo, *Concepción*, *Viña del Mar*) |
| `quote` y `escape` | `"` | Interpreta correctamente los valores entrecomillados con comillas internas duplicadas, como `"Monitor 27"""` |

**Decisión técnica (lectura como texto).** Si se infiriera el esquema, Spark convertiría en nulo cualquier valor que no calce con el tipo detectado (por ejemplo, un precio `$ 449990` en una columna numérica), y ese dato se perdería sin dejar rastro. Al leer todo como texto se conserva el valor original, y las conversiones de tipo se realizan en la Parte 2 de forma explícita y verificable.

Tras la carga, cada conteo se registra en la bitácora de trazabilidad.

In [0]:
def leer_csv(nombre_archivo):
    """Lee un CSV del volume con opciones idénticas para las tres sucursales."""
    return (spark.read
            .option("header", True)        # primera fila = nombres de columnas
            .option("inferSchema", False)  # todo como string; los tipos se corrigen en la Parte 2
            .option("encoding", "UTF-8")   # conserva tildes y ñ
            .option("quote", '"')          # carácter de entrecomillado
            .option("escape", '"')         # comillas internas duplicadas: "Monitor 27"""
            .csv(f"{RUTA_DATOS}/{nombre_archivo}"))

# Un DataFrame por sucursal
df_vina       = leer_csv("ventas_vina_del_mar.csv")
df_santiago   = leer_csv("ventas_santiago.csv")
df_concepcion = leer_csv("ventas_concepcion.csv")

# Conteo de filas de cada archivo, registrado en la bitácora
n_vina       = registrar("Carga Viña del Mar", df_vina)
n_santiago   = registrar("Carga Santiago", df_santiago)
n_concepcion = registrar("Carga Concepción", df_concepcion)

Carga Viña del Mar: 105 filas
Carga Santiago: 125 filas
Carga Concepción: 85 filas


%md
**Resultado.** Los tres archivos se cargaron sin errores en DataFrames independientes:

| DataFrame | Sucursal | Filas |
|---|---|---|
| `df_vina` | Viña del Mar | 105 |
| `df_santiago` | Santiago | 125 |
| `df_concepcion` | Concepción | 85 |

En conjunto suman $105 + 125 + 85 = 315$ registros. Este valor es la referencia para verificar la integridad de la unión en el paso 1.3: el DataFrame unificado debe contener exactamente $315$ filas, ni más ni menos.

%md
### 1.2 Inspección inicial de los DataFrames

Antes de unir los archivos se examina cada DataFrame por separado con dos métodos:

- `printSchema()`: muestra el nombre y el tipo de cada columna. Como la lectura se realizó sin inferencia de esquema, se espera que las nueve columnas sean de tipo `string`.
- `show(5, truncate=False)`: muestra las primeras cinco filas sin recortar el contenido de las celdas, lo que permite detectar a simple vista los problemas de formato que se abordarán en la Parte 2.

Esta inspección cumple además un propósito de control: confirmar que los tres archivos comparten exactamente la misma estructura, condición necesaria para integrarlos en un único DataFrame.

In [0]:
# Diccionario sucursal -> DataFrame, para inspeccionar los tres con el mismo código
dataframes = {
    "Viña del Mar": df_vina,
    "Santiago": df_santiago,
    "Concepción": df_concepcion,
}

for sucursal, df in dataframes.items():
    print("=" * 90)
    print(f"Sucursal: {sucursal}")
    print("=" * 90)
    df.printSchema()            # nombres y tipos de las columnas
    df.show(5, truncate=False)  # primeras 5 filas sin recortar el contenido

Sucursal: Viña del Mar
root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- producto_nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- sucursal: string (nullable = true)

+--------------+----------+-----------+-----------------+--------------+--------+---------------+----------+------------+
|id_transaccion|fecha     |id_producto|producto_nombre  |categoria     |cantidad|precio_unitario|id_cliente|sucursal    |
+--------------+----------+-----------+-----------------+--------------+--------+---------------+----------+------------+
|TRX10030      |01/10/2024|PROD889    |Mouse Inalámbrico|Almacenamiento|8       |79990          |CLI1158   |Viña del Mar|
|TRX10065      |2024/10/13|PROD403    |Teclado Mecánico |Almacenamiento|-1      |1              |CLI

%md
**Resultado.** Los tres DataFrames comparten exactamente la misma estructura: nueve columnas con idénticos nombres, en el mismo orden y de tipo `string`, como se esperaba al leer sin inferencia de esquema. Esto confirma que la integración en un único DataFrame es viable sin renombrar ni reordenar columnas.

Las primeras cinco filas de cada sucursal permiten anticipar los problemas de calidad que se tratarán en la Parte 2:

| Columna | Problema observado | Ejemplos en la muestra |
|---|---|---|
| `fecha` | Cuatro formatos distintos | `2024-11-05`, `2024/10/13`, `01/10/2024`, `10-11-2024` |
| `precio_unitario` | Símbolo monetario con y sin espacio | `79990`, `$349990`, `$ 349990` |
| `producto_nombre` | Mayúsculas y tildes inconsistentes | `Webcam HD` / `webcam hd`; `Impresora Láser` / `impresora laser` |
| `categoria` | Valores faltantes | `NULL` en las tres sucursales |
| `cantidad` | Valores no válidos | `-1` y `0` en Viña del Mar, `-1` en Santiago |
| `precio_unitario` | Valores implausibles para el rubro | `1` (TRX10065, Viña del Mar) |

Se observa además que la columna `sucursal` viene informada en cada archivo, por lo que no es necesario agregarla para identificar el origen de cada registro después de la unión.

%md
**Resultado.** Los tres DataFrames comparten exactamente la misma estructura: nueve columnas con idénticos nombres, en el mismo orden y de tipo `string`, como se esperaba al leer sin inferencia de esquema. Esto confirma que la integración en un único DataFrame es viable sin renombrar ni reordenar columnas.

Las primeras cinco filas de cada sucursal permiten anticipar los problemas de calidad que se tratarán en la Parte 2:

| Columna | Problema observado | Ejemplos en la muestra |
|---|---|---|
| `fecha` | Cuatro formatos distintos | `2024-11-05`, `2024/10/13`, `01/10/2024`, `10-11-2024` |
| `precio_unitario` | Símbolo monetario con y sin espacio | `79990`, `$349990`, `$ 349990` |
| `producto_nombre` | Mayúsculas y tildes inconsistentes | `Webcam HD` / `webcam hd`; `Impresora Láser` / `impresora laser` |
| `categoria` | Valores faltantes | `NULL` en las tres sucursales |
| `cantidad` | Valores no válidos | `-1` y `0` en Viña del Mar, `-1` en Santiago |
| `precio_unitario` | Valores implausibles para el rubro | `1` (TRX10065, Viña del Mar) |

Se observa además que la columna `sucursal` viene informada en cada archivo, por lo que no es necesario agregarla para identificar el origen de cada registro después de la unión.

In [0]:
# Control previo: los tres DataFrames deben tener las mismas columnas en el mismo orden
assert df_vina.columns == df_santiago.columns == df_concepcion.columns, \
    "Los archivos no comparten la misma estructura de columnas"

# Unión por nombre de columna (no por posición)
df_ventas_unificado = (df_vina
                       .unionByName(df_santiago)
                       .unionByName(df_concepcion))

# Control 1: el total unificado debe ser la suma exacta de los tres archivos
n_esperado  = n_vina + n_santiago + n_concepcion
n_unificado = registrar("Unión (df_ventas_unificado)", df_ventas_unificado)
assert n_unificado == n_esperado, \
    f"Error de integridad: se esperaban {n_esperado} filas y se obtuvieron {n_unificado}"
print(f"Verificación superada: {n_vina} + {n_santiago} + {n_concepcion} = {n_esperado} = {n_unificado}")

# Control 2: cada sucursal debe aportar exactamente las filas de su archivo
display(df_ventas_unificado.groupBy("sucursal").count().orderBy("sucursal"))

Unión (df_ventas_unificado): 315 filas
Verificación superada: 105 + 125 + 85 = 315 = 315


sucursal,count
Concepción,85
Santiago,125
Viña del Mar,105


%md
**Resultado.** La integración superó ambos controles:

1. **Total de filas:** $105 + 125 + 85 = 315 = n_{\text{unificado}}$. El `assert` no se activó, lo que confirma que la unión no creó ni eliminó registros.
2. **Aporte por sucursal:** Concepción $85$, Santiago $125$ y Viña del Mar $105$, idéntico a los conteos de cada archivo. Cada fila del DataFrame unificado conserva su sucursal de origen y ningún archivo se perdió ni se duplicó en la operación.

`df_ventas_unificado` contiene, por lo tanto, los $315$ registros originales sin pérdida ni duplicación atribuible a la integración, y constituye el punto de partida de la limpieza. Cabe precisar que los duplicados presentes *dentro* de los archivos fuente se conservan en esta etapa a propósito: se cuantifican y eliminan en la Parte 2, de modo que su efecto quede registrado en la bitácora.

%md
## Parte 2. Limpieza y depuración de datos (ID4.2)

### 2.0 Perfil de calidad inicial

Antes de aplicar cualquier transformación se mide la calidad de `df_ventas_unificado` con la función `perfil_calidad()`, que calcula tres indicadores:

| Indicador | Definición |
|---|---|
| Filas totales | Número de registros del DataFrame |
| Nulos por columna | Valores nulos o cadenas vacías (tras eliminar espacios) en cada columna |
| Registros duplicados exactos | $n_{\text{filas}} - n_{\text{filas distintas}}$, es decir, filas idénticas en todas sus columnas |

**Decisión técnica (medición reproducible).** El perfil se encapsula en una función para aplicar exactamente el mismo criterio al conjunto inicial y al final. De este modo, la comparación "antes/después" de la Parte 4 refleja únicamente el efecto de la limpieza y no diferencias en la forma de medir. El resultado se almacena en `perfil_antes`.

In [0]:
def perfil_calidad(df):
    """Devuelve un diccionario con filas totales, duplicados exactos y nulos por columna."""
    n_filas = df.count()

    # Nulos por columna: se cuentan los null y las cadenas vacías o con solo espacios
    nulos = (df.select([
                F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1)
                       .otherwise(0)).alias(c)
                for c in df.columns])
             .first().asDict())

    # Duplicados exactos: filas totales menos filas distintas
    n_duplicados = n_filas - df.dropDuplicates().count()

    perfil = {"Filas totales": n_filas, "Registros duplicados exactos": n_duplicados}
    for c in df.columns:
        perfil[f"Nulos en {c}"] = int(nulos[c])
    return perfil

# Perfil del conjunto unificado, antes de cualquier limpieza
perfil_antes = perfil_calidad(df_ventas_unificado)
display(spark.createDataFrame(list(perfil_antes.items()), ["indicador", "antes"]))

indicador,antes
Filas totales,315
Registros duplicados exactos,15
Nulos en id_transaccion,0
Nulos en fecha,0
Nulos en id_producto,0
Nulos en producto_nombre,0
Nulos en categoria,46
Nulos en cantidad,0
Nulos en precio_unitario,0
Nulos en id_cliente,0


%md
**Resultado.** El perfil inicial de `df_ventas_unificado` identifica dos problemas cuantificables:

| Indicador | Valor | Proporción |
|---|---|---|
| Filas totales | $315$ | $100\%$ |
| Registros duplicados exactos | $15$ | $15/315 \approx 4{,}8\%$ |
| Nulos en `categoria` | $46$ | $46/315 \approx 14{,}6\%$ |
| Nulos en las 8 columnas restantes | $0$ | $0\%$ |

Los valores faltantes se concentran exclusivamente en `categoria`; el resto de las columnas está completo. Este perfil no captura, sin embargo, los problemas de **formato** (fechas heterogéneas, precios con símbolo monetario, nombres inconsistentes) ni los valores **no válidos** (cantidades negativas o nulas), ya que en esos casos el dato existe pero es incorrecto. Esos problemas se tratan en los pasos 2.3 a 2.5 y en la Parte 3.

Estos valores quedan almacenados en `perfil_antes` y constituyen la línea base del informe comparativo de la Parte 4.

%md
### 2.1 Gestión de valores nulos en `categoria`

La columna `categoria` presenta $46$ valores nulos ($\approx 14{,}6\%$ del total). El enunciado plantea dos estrategias: eliminación (`dropna()`) o imputación (`fillna()`). Para elegir con fundamento se evalúan tres alternativas:

| Alternativa | Efecto | Evaluación |
|---|---|---|
| Eliminar las filas (`dropna`) | Se pierde el $14{,}6\%$ de las transacciones | Descartada: las filas son ventas reales con cantidad, precio y fecha válidos, cuya eliminación subestimaría los totales por sucursal |
| Imputar la categoría a partir del producto | Se asigna la categoría habitual de cada producto | Viable solo si cada producto pertenece a una única categoría; se verifica en esta celda |
| Imputar con una etiqueta explícita (`fillna`) | Las filas se conservan y quedan identificadas como sin categoría | Preserva la información sin inventar datos |

Para evaluar la segunda alternativa se cuenta, entre las filas con categoría informada, cuántas categorías distintas tiene cada producto. Los nombres se llevan a minúsculas solo para esta verificación; su estandarización definitiva se realiza en el paso 2.3.

In [0]:
# Conteo de nulos en categoria antes del tratamiento
n_nulos_cat = df_ventas_unificado.filter(F.col("categoria").isNull()).count()
print(f"Nulos en categoria: {n_nulos_cat}")

# Verificación: ¿cada producto tiene una única categoría? (solo filas con categoría informada)
evidencia_categoria = (df_ventas_unificado
    .filter(F.col("categoria").isNotNull())
    .groupBy(F.lower(F.trim(F.col("producto_nombre"))).alias("producto_minusculas"))
    .agg(F.countDistinct("categoria").alias("n_categorias"),
         F.array_sort(F.collect_set("categoria")).alias("categorias_observadas"))
    .orderBy("producto_minusculas"))
display(evidencia_categoria)

# Imputación con etiqueta explícita; df_limpio acumula las transformaciones de la Parte 2
df_limpio = df_ventas_unificado.fillna({"categoria": "Sin categoría"})

# Control: no deben quedar nulos en categoria
assert df_limpio.filter(F.col("categoria").isNull()).count() == 0
display(df_limpio.groupBy("categoria").count().orderBy(F.desc("count")))

Nulos en categoria: 46


producto_minusculas,n_categorias,categorias_observadas
auriculares bluetooth,5,"List(Almacenamiento, Audio, Computación, Periféricos, Redes)"
disco ssd 1tb,5,"List(Almacenamiento, Audio, Computación, Periféricos, Redes)"
impresora laser,5,"List(Almacenamiento, Audio, Computación, Periféricos, Redes)"
impresora láser,5,"List(Almacenamiento, Audio, Computación, Periféricos, Redes)"
laptop gamer,5,"List(Almacenamiento, Audio, Computación, Periféricos, Redes)"
laptop gaming,4,"List(Almacenamiento, Audio, Computación, Periféricos)"
memoria ram 16gb,5,"List(Almacenamiento, Audio, Computación, Periféricos, Redes)"
monitor 27 pulgadas,4,"List(Almacenamiento, Audio, Periféricos, Redes)"
"monitor 27""",5,"List(Almacenamiento, Audio, Computación, Periféricos, Redes)"
mouse inalambrico,5,"List(Almacenamiento, Audio, Computación, Periféricos, Redes)"


categoria,count
Periféricos,55
Computación,55
Audio,55
Almacenamiento,53
Redes,51
Sin categoría,46


%md
**Resultado.** La verificación descarta la imputación basada en el producto: de los $17$ nombres de producto observados, ninguno se asocia a una sola categoría. Cada uno aparece en entre $3$ y $5$ categorías distintas; por ejemplo, *webcam hd* figura en Almacenamiento, Audio, Computación, Periféricos y Redes. La categoría no es, por lo tanto, un atributo consistente del producto en estos datos, y cualquier regla de imputación basada en él asignaría un valor arbitrario.

**Decisión técnica.** Se imputan los $46$ nulos con la etiqueta explícita `"Sin categoría"` mediante `fillna()`. Esta estrategia:

1. conserva el $100\%$ de las transacciones, de modo que los totales de venta por sucursal no se subestiman;
2. no introduce información falsa, ya que la etiqueta declara que el dato no estaba disponible;
3. permite aislar estos registros en los análisis por categoría de la Parte 5.

Tras la imputación no quedan nulos en `categoria` (control `assert` superado). La distribución resultante es: Computación $55$, Audio $55$, Periféricos $55$, Almacenamiento $53$, Redes $51$ y Sin categoría $46$, con un total de $315$ filas, sin cambios en el volumen.

**Observación de calidad.** La asociación incoherente entre producto y categoría (por ejemplo, un teclado clasificado como "Audio") es en sí misma un problema de calidad de los datos fuente. Se documenta aquí y se retoma al interpretar la consulta por categoría de la Parte 5.

%md
**Resultado.** La verificación descarta la imputación basada en el producto: de los $17$ nombres de producto observados, ninguno se asocia a una sola categoría. Cada uno aparece en entre $3$ y $5$ categorías distintas; por ejemplo, *webcam hd* figura en Almacenamiento, Audio, Computación, Periféricos y Redes. La categoría no es, por lo tanto, un atributo consistente del producto en estos datos, y cualquier regla de imputación basada en él asignaría un valor arbitrario.

**Decisión técnica.** Se imputan los $46$ nulos con la etiqueta explícita `"Sin categoría"` mediante `fillna()`. Esta estrategia:

1. conserva el $100\%$ de las transacciones, de modo que los totales de venta por sucursal no se subestiman;
2. no introduce información falsa, ya que la etiqueta declara que el dato no estaba disponible;
3. permite aislar estos registros en los análisis por categoría de la Parte 5.

Tras la imputación no quedan nulos en `categoria` (control `assert` superado). La distribución resultante es: Computación $55$, Audio $55$, Periféricos $55$, Almacenamiento $53$, Redes $51$ y Sin categoría $46$, con un total de $315$ filas, sin cambios en el volumen.

**Observación de calidad.** La asociación incoherente entre producto y categoría (por ejemplo, un teclado clasificado como "Audio") es en sí misma un problema de calidad de los datos fuente. Se documenta aquí y se retoma al interpretar la consulta por categoría de la Parte 5.

In [0]:
n_antes_dup = df_limpio.count()

# 1. Identificación: filas que aparecen más de una vez (agrupando por todas las columnas)
duplicados = (df_limpio
              .groupBy(df_limpio.columns).count()
              .filter(F.col("count") > 1)
              .orderBy("sucursal", "id_transaccion"))
print("Registros duplicados encontrados:")
duplicados.select("id_transaccion", "fecha", "producto_nombre", "cantidad",
                  "precio_unitario", "sucursal", "count").show(truncate=False)

# 2. Eliminación: se conserva una ocurrencia de cada fila idéntica
df_limpio = df_limpio.dropDuplicates()

# 3. Verificación
n_despues_dup = registrar("Eliminación de duplicados", df_limpio)
print(f"Filas eliminadas: {n_antes_dup} - {n_despues_dup} = {n_antes_dup - n_despues_dup}")
assert n_antes_dup - n_despues_dup == perfil_antes["Registros duplicados exactos"]

n_ids = df_limpio.select("id_transaccion").distinct().count()
print(f"id_transaccion distintos: {n_ids} de {n_despues_dup} filas")
assert n_ids == n_despues_dup, "Existen identificadores repetidos con datos distintos"

print("Filas por sucursal tras eliminar duplicados:")
df_limpio.groupBy("sucursal").count().orderBy("sucursal").show()

Registros duplicados encontrados:
+--------------+----------+---------------------+--------+---------------+------------+-----+
|id_transaccion|fecha     |producto_nombre      |cantidad|precio_unitario|sucursal    |count|
+--------------+----------+---------------------+--------+---------------+------------+-----+
|TRX30000      |2024-10-03|Router WIFI          |9       |89990          |Concepción  |2    |
|TRX30018      |18/10/2024|teclado mecanico     |2       |$ 159990       |Concepción  |2    |
|TRX30022      |2024/12/26|Laptop Gamer         |2       |45990          |Concepción  |2    |
|TRX30030      |2024/11/23|Monitor 27 Pulgadas  |3       |$ 79990        |Concepción  |2    |
|TRX30031      |24/10/2024|auriculares bluetooth|8       |$ 89990        |Concepción  |2    |
|TRX20004      |2024/10/09|Mouse Inalambrico    |3       |$50            |Santiago    |2    |
|TRX20026      |2024/10/23|RAM 16GB             |8       |159990         |Santiago    |2    |
|TRX20044      |03/10/2024

%md
**Resultado.** Se identificaron y eliminaron $15$ registros duplicados exactos, cada uno repetido dos veces en su archivo de origen:

| Sucursal | Filas antes | Duplicados | Filas después |
|---|---|---|---|
| Concepción | $85$ | $5$ | $80$ |
| Santiago | $125$ | $5$ | $120$ |
| Viña del Mar | $105$ | $5$ | $100$ |
| **Total** | $315$ | $15$ | $300$ |

Los tres controles se cumplieron:

1. $315 - 300 = 15$, idéntico al número de duplicados del perfil inicial (`assert` superado).
2. Tras la eliminación hay $300$ identificadores distintos en $300$ filas: `id_transaccion` es ahora una clave única, y no existen transacciones con un mismo identificador y datos diferentes.
3. Cada sucursal perdió exactamente $5$ filas. La distribución uniforme sugiere una duplicación sistemática en la generación de los archivos, más que errores aislados de digitación.

Entre los duplicados figuran registros con otros problemas de calidad, como el precio $\$8.888.888$ (TRX20047), el precio $\$50$ (TRX20004) y la cantidad $0$ (TRX10053). Eliminar los duplicados antes de tratar esos problemas evita que cada error se contabilice dos veces en los indicadores posteriores. El resultado queda registrado en la bitácora: $315 \rightarrow 300$ filas.

%md
### 2.3 Estandarización de `producto_nombre`

Un mismo producto aparece escrito de varias formas, lo que impide agruparlo correctamente. Las inconsistencias son de dos tipos:

| Tipo | Ejemplos | Tratamiento |
|---|---|---|
| Formato | `TECLADO MECÁNICO`, `Teclado Mecánico`, `teclado mecanico` | `trim` (espacios extremos), `lower` (minúsculas), `translate` (eliminación de tildes) y `regexp_replace` (espacios múltiples) |
| Sinónimos | `Laptop Gaming` / `Laptop Gamer`; `SSD 1TB` / `Disco SSD 1TB`; `RAM 16GB` / `Memoria RAM 16GB`; `Monitor 27"` / `Monitor 27 Pulgadas` | Diccionario de equivalencias aplicado con `create_map` |

**Decisión técnica (más allá de las minúsculas).** El enunciado sugiere como ejemplo llevar los nombres a minúsculas. Esa transformación por sí sola es insuficiente: `teclado mecánico` y `teclado mecanico` seguirían siendo dos productos distintos, al igual que los sinónimos. Por ello se combina la normalización de formato con un diccionario explícito de equivalencias, construido a partir de los nombres observados en los datos. Se eliminan las tildes, en lugar de agregarlas, porque es la forma determinista de unificar: no requiere conocer la ortografía correcta de cada palabra.

Para verificar el resultado se conserva temporalmente el nombre original y se muestra qué variantes quedaron agrupadas bajo cada nombre estandarizado. Luego esa columna auxiliar se elimina.

In [0]:
from itertools import chain

n_variantes_antes = df_limpio.select("producto_nombre").distinct().count()

# Diccionario de sinónimos (clave: nombre ya normalizado; valor: nombre canónico)
sinonimos = {
    "laptop gaming": "laptop gamer",
    "ssd 1tb":       "disco ssd 1tb",
    "ram 16gb":      "memoria ram 16gb",
    'monitor 27"':   "monitor 27 pulgadas",
}
mapa_sinonimos = F.create_map([F.lit(x) for x in chain(*sinonimos.items())])

# Normalización de formato: trim -> minúsculas -> sin tildes -> un solo espacio entre palabras
nombre_base = F.regexp_replace(
    F.translate(F.lower(F.trim(F.col("producto_nombre"))), "áéíóúüñ", "aeiouun"),
    r"\s+", " ")

# Se aplica el sinónimo si existe; si no, se mantiene el nombre normalizado
df_limpio = (df_limpio
    .withColumn("producto_original", F.col("producto_nombre"))   # columna auxiliar de control
    .withColumn("producto_nombre", F.coalesce(mapa_sinonimos[nombre_base], nombre_base)))

n_variantes_despues = df_limpio.select("producto_nombre").distinct().count()
print(f"Variantes de nombre: {n_variantes_antes} antes -> {n_variantes_despues} después")

# Verificación: variantes originales agrupadas bajo cada nombre estandarizado
(df_limpio.groupBy("producto_nombre")
    .agg(F.count("*").alias("filas"),
         F.array_sort(F.collect_set("producto_original")).alias("variantes_originales"))
    .orderBy("producto_nombre")
    .show(truncate=False))

# Se elimina la columna auxiliar: el DataFrame conserva sus nueve columnas originales
df_limpio = df_limpio.drop("producto_original")

Variantes de nombre: 31 antes -> 10 después
+---------------------+-----+---------------------------------------------------------------------+
|producto_nombre      |filas|variantes_originales                                                 |
+---------------------+-----+---------------------------------------------------------------------+
|auriculares bluetooth|28   |[AURICULARES BLUETOOTH, Auriculares Bluetooth, auriculares bluetooth]|
|disco ssd 1tb        |32   |[Disco SSD 1TB, SSD 1TB, disco ssd 1tb]                              |
|impresora laser      |35   |[IMPRESORA LASER, Impresora Láser, impresora laser]                  |
|laptop gamer         |38   |[LAPTOP GAMER, Laptop Gamer, Laptop Gaming, laptop gamer]            |
|memoria ram 16gb     |22   |[Memoria RAM 16GB, RAM 16GB, memoria ram 16gb]                       |
|monitor 27 pulgadas  |28   |[Monitor 27 Pulgadas, Monitor 27", monitor 27 pulgadas]              |
|mouse inalambrico    |20   |[Mouse Inalambrico, Mouse I

%md
**Resultado.** La estandarización redujo las variantes de nombre de $31$ a $10$ productos, una reducción de $\frac{31-10}{31} \approx 67{,}7\%$. La tabla de verificación muestra que cada nombre estandarizado agrupa únicamente variantes del mismo producto, sin fusiones erróneas entre productos distintos:

- **Formato:** mayúsculas, tildes y combinaciones de ambas (por ejemplo, `TECLADO MECÁNICO`, `Teclado Mecánico` y `teclado mecanico` → `teclado mecanico`).
- **Sinónimos:** `Laptop Gaming` → `laptop gamer`, `SSD 1TB` → `disco ssd 1tb`, `RAM 16GB` → `memoria ram 16gb` y `Monitor 27"` → `monitor 27 pulgadas`.

La suma de filas de los $10$ productos es $28 + 32 + 35 + 38 + 22 + 28 + 20 + 34 + 32 + 31 = 300$, igual al total del DataFrame, por lo que la transformación no alteró el volumen de registros. El producto con más transacciones es *laptop gamer* ($38$) y el de menos es *mouse inalambrico* ($20$).

Sin este paso, cualquier agregación por producto habría fragmentado las ventas en $31$ grupos, subestimando el volumen real de cada artículo.

%md
### 2.4 Corrección de los formatos de `fecha`

La columna `fecha` es texto y combina varios formatos. Para convertirla a tipo `date` se procede en tres etapas:

1. **Diagnóstico:** cada dígito se reemplaza por `d` para obtener el *patrón* de cada valor (por ejemplo, `2024/10/13` → `dddd/dd/dd`), y se cuenta cuántas filas tiene cada patrón.
2. **Desambiguación día/mes:** en los formatos que comienzan con dos dígitos (`dd/dd/dddd` y `dd-dd-dddd`) hay que determinar si el primer número es el día o el mes. Si existen valores en que el primer número supera $12$, este solo puede ser el día.
3. **Conversión:** se asigna a cada patrón su formato con `when()` y se convierte con `to_date()`.

**Decisión técnica (`when` en lugar de intentos sucesivos).** Una alternativa habitual es aplicar `to_date()` con todos los formatos y quedarse con el primero que funcione (`coalesce`). En Databricks el modo ANSI está activo por defecto, y en ese modo `to_date()` lanza un error ante un valor que no calza con el formato, en vez de devolver nulo. Seleccionar el formato con una expresión regular evita ese problema y deja explícita la correspondencia entre cada patrón y su formato.

Se verifica finalmente que ninguna fecha quede nula y que todas pertenezcan al cuarto trimestre de 2024.

In [0]:
# 1. Diagnóstico: patrón de cada valor (dígitos reemplazados por "d")
print("Patrones de fecha encontrados:")
(df_limpio.groupBy(F.regexp_replace("fecha", r"\d", "d").alias("patron"))
    .agg(F.count("*").alias("filas"), F.min("fecha").alias("ejemplo"))
    .orderBy(F.desc("filas"))
    .show())

# 2. Desambiguación: en los formatos dd?dd?dddd, ¿el primer o el segundo número supera 12?
inicio_2_digitos = df_limpio.filter(F.col("fecha").rlike(r"^\d{2}[/-]\d{2}[/-]\d{4}$"))
n_primero_mayor_12 = inicio_2_digitos.filter(F.substring("fecha", 1, 2).cast("int") > 12).count()
n_segundo_mayor_12 = inicio_2_digitos.filter(F.substring("fecha", 4, 2).cast("int") > 12).count()
print(f"Formatos que comienzan con 2 dígitos: {inicio_2_digitos.count()} filas")
print(f"  primer número > 12:  {n_primero_mayor_12} filas  (solo puede ser el día)")
print(f"  segundo número > 12: {n_segundo_mayor_12} filas")

# 3. Conversión: cada patrón se asocia explícitamente a su formato
fecha_txt = F.trim(F.col("fecha"))
fecha_estandar = (F.when(fecha_txt.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(fecha_txt, "yyyy-MM-dd"))
                   .when(fecha_txt.rlike(r"^\d{4}/\d{2}/\d{2}$"), F.to_date(fecha_txt, "yyyy/MM/dd"))
                   .when(fecha_txt.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(fecha_txt, "dd/MM/yyyy"))
                   .when(fecha_txt.rlike(r"^\d{2}-\d{2}-\d{4}$"), F.to_date(fecha_txt, "dd-MM-yyyy")))

df_limpio = (df_limpio
    .withColumn("fecha_original", F.col("fecha"))   # columna auxiliar de control
    .withColumn("fecha", fecha_estandar))

# Verificación: sin nulos, rango dentro del cuarto trimestre de 2024 y ejemplos de conversión
n_fechas_nulas = df_limpio.filter(F.col("fecha").isNull()).count()
print(f"Fechas nulas tras la conversión: {n_fechas_nulas}")
assert n_fechas_nulas == 0, "Existen fechas con un formato no contemplado"

df_limpio.select(F.min("fecha").alias("fecha_minima"), F.max("fecha").alias("fecha_maxima")).show()
print("Ejemplo de conversión (un caso por patrón):")
(df_limpio.withColumn("patron", F.regexp_replace("fecha_original", r"\d", "d"))
    .groupBy("patron").agg(F.min("fecha_original").alias("fecha_original"))
    .join(df_limpio.select("fecha_original", "fecha").distinct(), "fecha_original")
    .select("patron", "fecha_original", "fecha").orderBy("patron")
    .show())

df_limpio = df_limpio.drop("fecha_original")
df_limpio.printSchema()


Patrones de fecha encontrados:
+----------+-----+----------+
|    patron|filas|   ejemplo|
+----------+-----+----------+
|dddd-dd-dd|   84|2024-10-01|
|dddd/dd/dd|   80|2024/10/01|
|dd/dd/dddd|   68|01/10/2024|
|dd-dd-dddd|   68|01-11-2024|
+----------+-----+----------+

Formatos que comienzan con 2 dígitos: 136 filas
  primer número > 12:  83 filas  (solo puede ser el día)
  segundo número > 12: 0 filas
Fechas nulas tras la conversión: 0
+------------+------------+
|fecha_minima|fecha_maxima|
+------------+------------+
|  2024-10-01|  2024-12-29|
+------------+------------+

Ejemplo de conversión (un caso por patrón):
+----------+--------------+----------+
|    patron|fecha_original|     fecha|
+----------+--------------+----------+
|dd-dd-dddd|    01-11-2024|2024-11-01|
|dd/dd/dddd|    01/10/2024|2024-10-01|
|dddd-dd-dd|    2024-10-01|2024-10-01|
|dddd/dd/dd|    2024/10/01|2024-10-01|
+----------+--------------+----------+

root
 |-- id_transaccion: string (nullable = true)
 |-- fec

%md
**Resultado.** La columna `fecha` presentaba cuatro formatos, que cubren la totalidad de los registros ($84 + 80 + 68 + 68 = 300$):

| Patrón | Formato asignado | Filas | Ejemplo original | Fecha convertida |
|---|---|---|---|---|
| `dddd-dd-dd` | `yyyy-MM-dd` | $84$ | `2024-10-01` | 2024-10-01 |
| `dddd/dd/dd` | `yyyy/MM/dd` | $80$ | `2024/10/01` | 2024-10-01 |
| `dd/dd/dddd` | `dd/MM/yyyy` | $68$ | `01/10/2024` | 2024-10-01 |
| `dd-dd-dddd` | `dd-MM-yyyy` | $68$ | `01-11-2024` | 2024-11-01 |

**Desambiguación día/mes.** De las $136$ fechas que comienzan con dos dígitos, $83$ tienen el primer número mayor que $12$ (por lo tanto, es el día) y ninguna tiene el segundo número mayor que $12$. La evidencia es concluyente: estos formatos siguen la convención día/mes/año, la habitual en Chile. Interpretarlos como mes/día habría producido fechas inválidas en $83$ casos y fechas erróneas en el resto.

**Verificación.** Tras la conversión no hay fechas nulas (`assert` superado) y el esquema confirma que `fecha` es ahora de tipo `date`. El rango va del $1$ de octubre al $29$ de diciembre de 2024, en concordancia con el período del caso de estudio (cuarto trimestre). El tipo `date` habilita las operaciones temporales de la Parte 3, como la obtención del día de la semana.

%md
### 2.5 Limpieza y conversión de `precio_unitario` (y tipado de `cantidad`)

`precio_unitario` es texto y mezcla valores con y sin símbolo monetario (`449990`, `$449990`, `$ 449990`), por lo que no admite operaciones aritméticas. Se procede en cuatro etapas:

1. **Diagnóstico:** se cuenta cuántas filas tiene cada formato.
2. **Limpieza:** `regexp_replace` elimina el símbolo `$` y los espacios mediante la expresión regular `[\$\s]`.
3. **Control previo a la conversión:** se verifica con `rlike("^\d+$")` que, tras la limpieza, todos los valores estén compuestos exclusivamente por dígitos. Esto es necesario porque, en modo ANSI, convertir a número un texto no numérico detiene la ejecución.
4. **Conversión:** `cast("double")` transforma el precio a tipo numérico.

En la misma celda se convierte `cantidad` a entero (`int`), ya que el filtrado de la Parte 3 ($\text{cantidad} \le 0$) y el cálculo de $\text{precio\_total}$ requieren que sea numérica.

**Decisión técnica (separadores).** Los precios están expresados en pesos chilenos enteros, sin separador de miles ni decimales; por eso basta con eliminar el símbolo y los espacios, sin necesidad de interpretar puntos o comas.

In [0]:
# 1. Diagnóstico: formatos presentes en precio_unitario
formato_precio = (F.when(F.col("precio_unitario").startswith("$ "), "con '$ ' (símbolo y espacio)")
                   .when(F.col("precio_unitario").startswith("$"),  "con '$' (símbolo sin espacio)")
                   .otherwise("solo dígitos"))
print("Formatos de precio_unitario:")
(df_limpio.groupBy(formato_precio.alias("formato")).count()
    .orderBy(F.desc("count")).show(truncate=False))

# 2. Limpieza: se eliminan el símbolo $ y todos los espacios
precio_txt = F.regexp_replace(F.col("precio_unitario"), r"[\$\s]", "")

# 3. Control: tras la limpieza, todos los valores deben ser solo dígitos
n_no_numericos = df_limpio.filter(~precio_txt.rlike(r"^\d+$")).count()
print(f"Valores no numéricos tras la limpieza: {n_no_numericos}")
assert n_no_numericos == 0, "Existen precios con caracteres no contemplados"

# 4. Conversión de tipos
df_limpio = (df_limpio
    .withColumn("precio_unitario", precio_txt.cast("double"))
    .withColumn("cantidad", F.trim(F.col("cantidad")).cast("int")))

# Verificación: ningún valor se perdió en la conversión y estadísticos descriptivos
n_nulos_num = df_limpio.filter(F.col("precio_unitario").isNull() | F.col("cantidad").isNull()).count()
print(f"Nulos en precio_unitario o cantidad tras la conversión: {n_nulos_num}")
assert n_nulos_num == 0

print("Estadísticos descriptivos:")
df_limpio.select("precio_unitario", "cantidad").summary("count", "min", "25%", "50%", "75%", "max").show()
df_limpio.printSchema()

Formatos de precio_unitario:
+-----------------------------+-----+
|formato                      |count|
+-----------------------------+-----+
|con '$ ' (símbolo y espacio) |110  |
|solo dígitos                 |108  |
|con '$' (símbolo sin espacio)|82   |
+-----------------------------+-----+

Valores no numéricos tras la limpieza: 0
Nulos en precio_unitario o cantidad tras la conversión: 0
Estadísticos descriptivos:
+-------+---------------+--------+
|summary|precio_unitario|cantidad|
+-------+---------------+--------+
|  count|            300|     300|
|    min|            1.0|      -5|
|    25%|        79990.0|       2|
|    50%|       129990.0|       5|
|    75%|       299990.0|       8|
|    max|      9999990.0|      10|
+-------+---------------+--------+

root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- producto_nombre: string (nullable = true)
 |-- categoria: string (nullable = false)
 |-- cant

%md
**Resultado.** `precio_unitario` presentaba tres formatos: $110$ valores con símbolo y espacio (`$ 449990`), $108$ solo con dígitos y $82$ con símbolo sin espacio (`$449990`). Es decir, el $\frac{192}{300} = 64\%$ de los precios no era numérico en origen. Tras la limpieza, todos los valores quedaron compuestos solo por dígitos, la conversión no generó nulos (ambos `assert` superados) y el esquema confirma los tipos `double` para `precio_unitario` e `integer` para `cantidad`.

Los estadísticos descriptivos revelan, sin embargo, problemas de **validez** que la conversión de tipo no resuelve:

| Variable | Mín. | $Q_1$ | Mediana | $Q_3$ | Máx. | Observación |
|---|---|---|---|---|---|---|
| `precio_unitario` | $1$ | $79.990$ | $129.990$ | $299.990$ | $9.999.990$ | Extremos implausibles para productos tecnológicos |
| `cantidad` | $-5$ | $2$ | $5$ | $8$ | $10$ | Existen cantidades negativas |

- El rango central de precios ($Q_1$ a $Q_3$) es coherente con el rubro, pero el mínimo de $\$1$ y el máximo de $\$9.999.990$ sugieren valores de relleno o errores de digitación. Su detección formal mediante el rango intercuartílico se realiza en la Parte 4.
- Las cantidades negativas (y las nulas) no representan ventas válidas y se filtran en la Parte 3.

Con este paso concluye la Parte 2: `df_limpio` tiene $300$ filas, sin duplicados ni nulos, con nombres de producto estandarizados, fechas de tipo `date` y precios y cantidades numéricos.

%md
## Parte 3. Transformación y análisis con DataFrames (ID4.1)

### 3.1 Filtrado de registros con cantidad no válida

Una venta válida implica al menos una unidad vendida. Los registros con $\text{cantidad} \le 0$ no representan ventas: los valores negativos corresponden posiblemente a devoluciones o anulaciones registradas como transacciones, y los valores cero, a errores de digitación o transacciones incompletas. Incluirlos distorsionaría los indicadores de venta, ya que un valor negativo restaría ingresos al calcular $\text{precio\_total}$.

**Decisión técnica (filtrar en lugar de corregir).** No existe información que permita reconstruir la cantidad correcta (por ejemplo, convertir $-2$ en $2$ supondría que el signo es un error, sin evidencia que lo respalde). Por ello, siguiendo el enunciado, estos registros se excluyen con `.filter()`. Antes de filtrar se caracterizan por tipo y sucursal para documentar qué se elimina, y después se verifica el conteo:

$$
n_{\text{después}} = n_{\text{antes}} - n_{\text{cantidad} \le 0}
$$

A partir de este paso el DataFrame se denomina `df_ventas`, para distinguir el conjunto de ventas válidas del resultado de la limpieza de la Parte 2.

In [0]:
n_antes_filtro = df_limpio.count()

# Caracterización de los registros no válidos antes de eliminarlos
invalidas = df_limpio.filter(F.col("cantidad") <= 0)
tipo_invalida = F.when(F.col("cantidad") < 0, "negativa").otherwise("cero")
n_invalidas = invalidas.count()
print(f"Registros con cantidad <= 0: {n_invalidas}")
invalidas.groupBy(tipo_invalida.alias("tipo")).count().orderBy("tipo").show()
invalidas.groupBy("sucursal", tipo_invalida.alias("tipo")).count().orderBy("sucursal", "tipo").show()

# Filtrado: se conservan solo las ventas con al menos una unidad
df_ventas = df_limpio.filter(F.col("cantidad") > 0)

# Verificación del conteo
n_despues_filtro = registrar("Filtro cantidad > 0", df_ventas)
print(f"Verificación: {n_antes_filtro} - {n_invalidas} = {n_antes_filtro - n_invalidas} = {n_despues_filtro}")
assert n_despues_filtro == n_antes_filtro - n_invalidas
assert df_ventas.filter(F.col("cantidad") <= 0).count() == 0

print("Filas por sucursal tras el filtrado:")
df_ventas.groupBy("sucursal").count().orderBy("sucursal").show()

Registros con cantidad <= 0: 26
+--------+-----+
|    tipo|count|
+--------+-----+
|    cero|    4|
|negativa|   22|
+--------+-----+

+------------+--------+-----+
|    sucursal|    tipo|count|
+------------+--------+-----+
|  Concepción|    cero|    2|
|  Concepción|negativa|    8|
|    Santiago|negativa|    8|
|Viña del Mar|    cero|    2|
|Viña del Mar|negativa|    6|
+------------+--------+-----+

Filtro cantidad > 0: 274 filas
Verificación: 300 - 26 = 274 = 274
Filas por sucursal tras el filtrado:
+------------+-----+
|    sucursal|count|
+------------+-----+
|  Concepción|   70|
|    Santiago|  112|
|Viña del Mar|   92|
+------------+-----+



%md
**Resultado.** Se identificaron y excluyeron $26$ registros con cantidad no válida, equivalentes a $\frac{26}{300} \approx 8{,}7\%$ del conjunto limpio:

| Sucursal | Cantidad negativa | Cantidad cero | Total excluido | Filas antes | Filas después |
|---|---|---|---|---|---|
| Concepción | $8$ | $2$ | $10$ | $80$ | $70$ |
| Santiago | $8$ | $0$ | $8$ | $120$ | $112$ |
| Viña del Mar | $6$ | $2$ | $8$ | $100$ | $92$ |
| **Total** | $22$ | $4$ | $26$ | $300$ | $274$ |

La verificación se cumplió: $300 - 26 = 274$, y no queda ningún registro con $\text{cantidad} \le 0$ (ambos `assert` superados). Predominan las cantidades negativas ($22$ de $26$), compatibles con devoluciones registradas como ventas. Concepción es la sucursal más afectada, pues pierde el $\frac{10}{80} = 12{,}5\%$ de sus registros, frente al $\frac{8}{120} \approx 6{,}7\%$ de Santiago y el $\frac{8}{100} = 8\%$ de Viña del Mar.

`df_ventas` contiene $274$ ventas válidas, y la bitácora registra $300 \rightarrow 274$ filas.

%md
### 3.2 Creación de columnas calculadas

Se agregan dos columnas derivadas:

| Columna | Definición | Función de PySpark |
|---|---|---|
| `precio_total` | $\text{precio\_total} = \text{cantidad} \times \text{precio\_unitario}$ | Producto de columnas |
| `dia_semana` | Nombre del día de la semana de la fecha de venta | `dayofweek()` y un diccionario de nombres |

Se agrega además la columna auxiliar `dia_semana_num` ($1$ = lunes, …, $7$ = domingo), que permite ordenar los días cronológicamente en los análisis; ordenados alfabéticamente aparecerían en un orden sin sentido.

**Decisión técnica (nombres en español).** `date_format(fecha, "EEEE")` devuelve el nombre del día en inglés, ya que depende de la configuración regional del clúster. Para que el resultado sea independiente de esa configuración y coherente con el idioma del informe, se obtiene el número del día con `dayofweek()` (que en Spark asigna $1$ al domingo y $7$ al sábado) y se traduce mediante un diccionario. Para la numeración de lunes a domingo se usa la transformación

$$
\text{dia\_semana\_num} = \big((\text{dayofweek} + 5) \bmod 7\big) + 1
$$

La corrección de ambas columnas se verifica comprobando a mano tres registros.

In [0]:
from itertools import chain

# Diccionario dayofweek de Spark (1 = domingo, ..., 7 = sábado) -> nombre en español
dias = {1: "domingo", 2: "lunes", 3: "martes", 4: "miércoles",
        5: "jueves", 6: "viernes", 7: "sábado"}
mapa_dias = F.create_map([F.lit(x) for x in chain(*dias.items())])

df_ventas = (df_ventas
    .withColumn("precio_total",   F.col("cantidad") * F.col("precio_unitario"))
    .withColumn("dia_semana_num", ((F.dayofweek("fecha") + 5) % 7) + 1)   # 1 = lunes ... 7 = domingo
    .withColumn("dia_semana",     mapa_dias[F.dayofweek("fecha")]))

# Control: las nuevas columnas no deben tener nulos
n_nulos_calc = df_ventas.filter(F.col("precio_total").isNull() | F.col("dia_semana").isNull()).count()
print(f"Nulos en las columnas calculadas: {n_nulos_calc}")
assert n_nulos_calc == 0

# Verificación manual de tres registros
print("Verificación de tres registros:")
(df_ventas.select("id_transaccion", "fecha", "dia_semana", "cantidad", "precio_unitario", "precio_total")
    .orderBy("id_transaccion").show(3))

# Distribución de las ventas por día de la semana (orden cronológico)
print("Número de ventas por día de la semana:")
(df_ventas.groupBy("dia_semana_num", "dia_semana").count()
    .orderBy("dia_semana_num").show())
df_ventas.printSchema()

Nulos en las columnas calculadas: 0
Verificación de tres registros:
+--------------+----------+----------+--------+---------------+------------+
|id_transaccion|     fecha|dia_semana|cantidad|precio_unitario|precio_total|
+--------------+----------+----------+--------+---------------+------------+
|      TRX10000|2024-12-21|    sábado|       2|        79990.0|    159980.0|
|      TRX10001|2024-10-04|   viernes|       9|        25990.0|    233910.0|
|      TRX10002|2024-10-01|    martes|       6|        45990.0|    275940.0|
+--------------+----------+----------+--------+---------------+------------+
only showing top 3 rows
Número de ventas por día de la semana:
+--------------+----------+-----+
|dia_semana_num|dia_semana|count|
+--------------+----------+-----+
|             1|     lunes|   27|
|             2|    martes|   41|
|             3| miércoles|   39|
|             4|    jueves|   40|
|             5|   viernes|   43|
|             6|    sábado|   45|
|             7|   domin

%md
**Resultado.** Las dos columnas calculadas se crearon sin nulos (`assert` superado) y el esquema las incorpora con los tipos esperados: `precio_total` como `double` y `dia_semana` como `string`, además de la auxiliar `dia_semana_num` como `integer`.

**Verificación manual:**

| Transacción | Fecha | Día calculado | Comprobación de $\text{precio\_total}$ |
|---|---|---|---|
| TRX10000 | 2024-12-21 | sábado | $2 \times 79.990 = 159.980$ |
| TRX10001 | 2024-10-04 | viernes | $9 \times 25.990 = 233.910$ |
| TRX10002 | 2024-10-01 | martes | $6 \times 45.990 = 275.940$ |

Los tres productos coinciden con el valor calculado, y los días de la semana corresponden al calendario de 2024.

**Distribución semanal.** Las $274$ ventas se reparten entre los siete días ($27 + 41 + 39 + 40 + 43 + 45 + 39 = 274$), con un promedio de $\frac{274}{7} \approx 39{,}1$ ventas por día. El sábado concentra el mayor número de transacciones ($45$) y el lunes el menor ($27$), un patrón consistente con el comercio minorista, donde el fin de semana suele tener mayor afluencia. Dado el tamaño reducido de la muestra (un trimestre, con unas $13$ ocurrencias de cada día de la semana), esta diferencia debe interpretarse como indicio y no como una tendencia concluyente.

%md
### 3.3 Análisis de valores únicos con `distinct()`

Una vez concluidas la limpieza y el filtrado, se utiliza `distinct()` para contar y listar los valores únicos de `sucursal` y `categoria` en `df_ventas`. Este análisis cumple dos funciones:

1. **Control de consistencia:** debe haber exactamente $3$ sucursales. Un número mayor indicaría variantes de escritura de una misma sucursal (por ejemplo, `Concepcion` y `Concepción`) que requerirían estandarización.
2. **Caracterización del dominio:** se espera encontrar las $5$ categorías originales más la etiqueta `"Sin categoría"` introducida en el paso 2.1, es decir, $6$ valores únicos.

Como complemento, se informa el número de ventas válidas por categoría y el número de productos únicos tras la estandarización del paso 2.3.

In [0]:
# Sucursales únicas
sucursales_unicas = df_ventas.select("sucursal").distinct()
n_sucursales = sucursales_unicas.count()
print(f"Sucursales únicas: {n_sucursales}")
sucursales_unicas.orderBy("sucursal").show()
assert n_sucursales == 3, "Existen variantes de escritura en sucursal"

# Categorías únicas
categorias_unicas = df_ventas.select("categoria").distinct()
n_categorias = categorias_unicas.count()
print(f"Categorías únicas: {n_categorias}")
categorias_unicas.orderBy("categoria").show()

# Complemento: ventas válidas por categoría y productos únicos
print("Ventas válidas por categoría:")
df_ventas.groupBy("categoria").count().orderBy(F.desc("count")).show()
print(f"Productos únicos: {df_ventas.select('producto_nombre').distinct().count()}")


Sucursales únicas: 3
+------------+
|    sucursal|
+------------+
|  Concepción|
|    Santiago|
|Viña del Mar|
+------------+

Categorías únicas: 6
+--------------+
|     categoria|
+--------------+
|Almacenamiento|
|         Audio|
|   Computación|
|   Periféricos|
|         Redes|
| Sin categoría|
+--------------+

Ventas válidas por categoría:
+--------------+-----+
|     categoria|count|
+--------------+-----+
|         Audio|   50|
|Almacenamiento|   49|
|   Computación|   48|
|   Periféricos|   45|
|         Redes|   42|
| Sin categoría|   40|
+--------------+-----+

Productos únicos: 10


%md
**Resultado.** El análisis con `distinct()` confirma la consistencia del conjunto limpio:

- **Sucursales:** $3$ valores únicos (Concepción, Santiago y Viña del Mar), sin variantes de escritura (`assert` superado). La integración de la Parte 1 preservó correctamente la identidad de cada sucursal.
- **Categorías:** $6$ valores únicos, es decir, las $5$ categorías originales más `"Sin categoría"`, tal como se esperaba tras la imputación del paso 2.1.
- **Productos:** $10$ productos únicos, consistente con la estandarización del paso 2.3. El filtrado no eliminó ningún producto completo del catálogo.

**Ventas válidas por categoría:**

| Categoría | Ventas | Proporción |
|---|---|---|
| Audio | $50$ | $18{,}2\%$ |
| Almacenamiento | $49$ | $17{,}9\%$ |
| Computación | $48$ | $17{,}5\%$ |
| Periféricos | $45$ | $16{,}4\%$ |
| Redes | $42$ | $15{,}3\%$ |
| Sin categoría | $40$ | $14{,}6\%$ |
| **Total** | $274$ | $100\%$ |

La distribución entre categorías es notablemente homogénea (entre $42$ y $50$ ventas). Ese reparto casi uniforme es coherente con la asignación aparentemente aleatoria de categorías observada en el paso 2.1. Los registros sin categoría bajaron de $46$ a $40$ al eliminar duplicados y cantidades no válidas, pero mantienen su peso relativo ($\approx 14{,}6\%$), lo que confirma que habría sido inadecuado descartarlos.

Con este paso concluye la Parte 3.

%md
## Parte 4. Validación y control de calidad (ID4.3)

### 4.1 Regla de validación: $\text{precio\_total} \ge 0$

Un monto de venta negativo carece de sentido económico en este contexto. La regla se formaliza como:

$$
R_1: \quad \text{precio\_total}_i \ge 0 \quad \forall\, i
$$

Dado que $\text{precio\_total} = \text{cantidad} \times \text{precio\_unitario}$, un valor negativo solo podría originarse en una cantidad negativa o en un precio negativo. La Parte 3 eliminó las cantidades no positivas y la limpieza del paso 2.5 dejó únicamente dígitos en los precios, por lo que se espera que la regla se cumpla en el $100\%$ de los registros.

**Decisión técnica (regla explícita aunque se espere su cumplimiento).** La validación no se omite por el hecho de que el resultado sea predecible. Una regla de calidad documenta y garantiza una propiedad del conjunto de datos: si en una ejecución futura cambiaran los archivos fuente o el orden de las transformaciones, el `assert` detendría el flujo antes de que un valor inválido llegue al análisis. La regla se implementa como una columna booleana `regla_precio_total_ok`, que deja el resultado de la validación registrado en cada fila.

In [0]:
# Regla R1: precio_total no negativo, registrada como columna booleana por fila
df_ventas = df_ventas.withColumn("regla_precio_total_ok", F.col("precio_total") >= 0)

n_total     = df_ventas.count()
n_cumple_r1 = df_ventas.filter(F.col("regla_precio_total_ok")).count()
n_viola_r1  = n_total - n_cumple_r1

print(f"Regla R1 (precio_total >= 0): {n_cumple_r1} de {n_total} registros cumplen; {n_viola_r1} la violan")
assert n_viola_r1 == 0, f"{n_viola_r1} registros con precio_total negativo"

# Rango de precio_total, como evidencia complementaria
df_ventas.select(F.min("precio_total").alias("precio_total_minimo"),
                 F.max("precio_total").alias("precio_total_maximo")).show()

Regla R1 (precio_total >= 0): 274 de 274 registros cumplen; 0 la violan
+-------------------+-------------------+
|precio_total_minimo|precio_total_maximo|
+-------------------+-------------------+
|                1.0|        7.9999992E7|
+-------------------+-------------------+



%md
**Resultado.** La regla $R_1$ se cumple en los $274$ registros ($100\%$) y no se detectaron violaciones (`assert` superado). La columna `regla_precio_total_ok` queda incorporada a `df_ventas` como evidencia de la validación a nivel de fila.

El rango observado de $\text{precio\_total}$ va desde $\$1$ hasta $\$79.999.992$ (Spark muestra este último en notación científica, `7.9999992E7`). Ambos extremos son llamativos para ventas de productos tecnológicos: una venta total de $\$1$ y otra cercana a $\$80$ millones no son plausibles en una tienda minorista. La regla $R_1$ verifica el **signo** del monto, pero no su **magnitud**, por lo que estos valores la superan sin problema. Su origen está en los precios unitarios extremos identificados en el paso 2.5, que se evalúan formalmente en el paso siguiente mediante el rango intercuartílico.

%md
### 4.2 Detección de valores atípicos en `precio_unitario` mediante el rango intercuartílico

Se aplica el criterio de Tukey basado en el rango intercuartílico (IQR). Con $Q_1$ y $Q_3$ como los percentiles $25$ y $75$:

$$
\text{IQR} = Q_3 - Q_1, \qquad
L_{\text{inf}} = Q_1 - 1{,}5 \cdot \text{IQR}, \qquad
L_{\text{sup}} = Q_3 + 1{,}5 \cdot \text{IQR}
$$

Un registro es atípico si $\text{precio\_unitario} < L_{\text{inf}}$ o $\text{precio\_unitario} > L_{\text{sup}}$. Los cuartiles se calculan con `approxQuantile()` con error relativo $0$, lo que produce el valor exacto. Para $274$ registros el costo computacional es despreciable.

**Decisión técnica (marcar, no eliminar).** Los registros atípicos se **marcan** con la columna booleana `es_outlier_iqr` y además se **separan** en el DataFrame `df_outliers`, como permite el enunciado. No se eliminan de `df_ventas`, porque un precio alto puede corresponder a un producto legítimamente caro; la decisión de excluirlos corresponde al análisis de negocio, que dispondrá de la marca para filtrarlos. Para cuantificar su relevancia se calcula la proporción de las ventas totales que representan.

**Regla complementaria.** Si la distribución tiene asimetría positiva, $L_{\text{inf}}$ puede resultar negativo; en ese caso el criterio IQR es incapaz de detectar precios anormalmente bajos. Para cubrir ese caso se define la regla de negocio
$$
R_2: \quad \text{precio\_unitario} \ge 1.000
$$
ya que ningún producto tecnológico del catálogo tiene un precio unitario inferior a mil pesos. Los registros que la violan se marcan con `precio_bajo_sospechoso`.

In [0]:
# Cuartiles exactos (error relativo 0) y límites de Tukey
q1, q3 = df_ventas.approxQuantile("precio_unitario", [0.25, 0.75], 0.0)
iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr
print(f"Q1 = {q1:,.0f} | Q3 = {q3:,.0f} | IQR = {iqr:,.0f}")
print(f"Límite inferior = {lim_inf:,.0f} | Límite superior = {lim_sup:,.0f}")

# Marcas por fila: outlier IQR y precio bajo sospechoso (regla R2)
df_ventas = (df_ventas
    .withColumn("es_outlier_iqr", (F.col("precio_unitario") < lim_inf) | (F.col("precio_unitario") > lim_sup))
    .withColumn("precio_bajo_sospechoso", F.col("precio_unitario") < 1000))

# Separación de los registros atípicos en su propio DataFrame
df_outliers = df_ventas.filter(F.col("es_outlier_iqr"))
print(f"\nOutliers IQR: {df_outliers.count()} de {df_ventas.count()} registros")
(df_outliers.select("id_transaccion", "sucursal", "producto_nombre", "cantidad", "precio_unitario", "precio_total")
    .orderBy(F.desc("precio_unitario")).show(truncate=False))

# Regla R2: precios bajo 1.000 (no detectables por IQR cuando el límite inferior es negativo)
df_precio_bajo = df_ventas.filter(F.col("precio_bajo_sospechoso"))
print(f"Precios bajo 1.000 (regla R2): {df_precio_bajo.count()} registros")
(df_precio_bajo.select("id_transaccion", "sucursal", "producto_nombre", "cantidad", "precio_unitario")
    .orderBy("precio_unitario").show(truncate=False))

# Resumen por sucursal
print("Registros marcados por sucursal:")
(df_ventas.groupBy("sucursal")
    .agg(F.count("*").alias("ventas"),
         F.sum(F.col("es_outlier_iqr").cast("int")).alias("outliers_iqr"),
         F.sum(F.col("precio_bajo_sospechoso").cast("int")).alias("precio_bajo"))
    .orderBy("sucursal").show())

# Peso de los outliers en el monto total de ventas
totales = df_ventas.agg(
    F.sum("precio_total").alias("total"),
    F.sum(F.when(F.col("es_outlier_iqr"), F.col("precio_total")).otherwise(0)).alias("total_outliers")).first()
print(f"Ventas totales:          ${totales['total']:,.0f}")
print(f"Ventas de los outliers:  ${totales['total_outliers']:,.0f}")
print(f"Proporción:              {totales['total_outliers'] / totales['total']:.1%}")

Q1 = 79,990 | Q3 = 299,990 | IQR = 220,000
Límite inferior = -250,010 | Límite superior = 629,990

Outliers IQR: 7 de 274 registros
+--------------+------------+---------------------+--------+---------------+------------+
|id_transaccion|sucursal    |producto_nombre      |cantidad|precio_unitario|precio_total|
+--------------+------------+---------------------+--------+---------------+------------+
|TRX20119      |Santiago    |auriculares bluetooth|7       |9999990.0      |6.999993E7  |
|TRX20037      |Santiago    |mouse inalambrico    |3       |9999990.0      |2.999997E7  |
|TRX20001      |Santiago    |disco ssd 1tb        |7       |9999990.0      |6.999993E7  |
|TRX30009      |Concepción  |impresora laser      |2       |9999990.0      |1.999998E7  |
|TRX30045      |Concepción  |laptop gamer         |8       |9999990.0      |7.999992E7  |
|TRX10037      |Viña del Mar|webcam hd            |4       |8888888.0      |3.5555552E7 |
|TRX20047      |Santiago    |router wifi          |9      

%md
**Resultado.** Los cuartiles de `precio_unitario` son $Q_1 = 79.990$ y $Q_3 = 299.990$, con $\text{IQR} = 220.000$. Los límites de Tukey resultan:

$$
L_{\text{inf}} = 79.990 - 1{,}5 \cdot 220.000 = -250.010, \qquad
L_{\text{sup}} = 299.990 + 1{,}5 \cdot 220.000 = 629.990
$$

**Outliers por IQR ($7$ registros, $\frac{7}{274} \approx 2{,}6\%$).** Todos superan el límite superior y corresponden a solo dos valores: $\$9.999.990$ ($5$ casos) y $\$8.888.888$ ($2$ casos). La repetición exacta de cifras con dígitos idénticos, asignadas a productos de naturaleza muy distinta (un mouse, una impresora, un router), indica que no se trata de productos caros, sino de **valores de relleno** o errores de carga. Ningún producto del catálogo tiene un precio regular superior a $\$449.990$.

**Precios anormalmente bajos (regla $R_2$, $7$ registros).** Cuatro ventas con precio $\$1$ y tres con precio $\$50$, entre ellas dos *laptop gamer* a $\$1$. Como $L_{\text{inf}} < 0$, el criterio IQR no podía detectarlos; la regla complementaria cubre esa limitación, que es estructural del método cuando la distribución tiene asimetría positiva.

| Sucursal | Ventas | Outliers IQR | Precio bajo | Total marcado |
|---|---|---|---|---|
| Concepción | $70$ | $2$ | $2$ | $4$ |
| Santiago | $112$ | $4$ | $3$ | $7$ |
| Viña del Mar | $92$ | $1$ | $2$ | $3$ |
| **Total** | $274$ | $7$ | $7$ | $14$ |

**Impacto en el monto de ventas.** Los $7$ outliers suman $\$385.555.274$ de un total de $\$639.025.343$, es decir, el $60{,}3\%$ de las ventas proviene del $2{,}6\%$ de los registros. Este es el hallazgo de calidad más relevante del notebook: cualquier indicador de ventas calculado sin considerar estas marcas estaría dominado por datos erróneos. Las columnas `es_outlier_iqr` y `precio_bajo_sospechoso` permiten al análisis posterior incluir o excluir estos registros de forma explícita.

%md
### 4.3 Informe de calidad comparativo (antes/después) y bitácora de trazabilidad

El informe compara el conjunto inicial (`df_ventas_unificado`, $315$ filas sin tratar) con el conjunto final (`df_ventas`) en dos bloques de indicadores:

1. **Indicadores de completitud y unicidad**, calculados con la misma función `perfil_calidad()` del paso 2.0: filas totales, registros duplicados exactos y nulos por columna. Para que la comparación sea homogénea, el perfil final se calcula solo sobre las nueve columnas originales, excluyendo las columnas derivadas y de control creadas en las Partes 3 y 4.
2. **Indicadores de validez y consistencia**, que capturan los problemas que el conteo de nulos no detecta: fechas no estandarizadas, precios no numéricos, cantidades no válidas y variantes de nombre de producto.

Se presentan además los registros **marcados** (no eliminados) por las reglas de la Parte 4 y, finalmente, la **bitácora de trazabilidad**, que reconstruye el número de filas después de cada operación que lo modificó.

**Decisión técnica (bitácora robusta a reejecuciones).** La bitácora es una lista que crece cada vez que se ejecuta `registrar()`. Si una celda se ejecutó más de una vez durante el desarrollo, un mismo paso podría aparecer repetido. Para evitarlo, se conserva una sola entrada por paso (el último conteo registrado), respetando el orden en que los pasos ocurrieron.

In [0]:
columnas_originales = df_ventas_unificado.columns

# Bloque 1: completitud y unicidad (misma función que en el paso 2.0, sobre las 9 columnas originales)
perfil_despues = perfil_calidad(df_ventas.select(columnas_originales))

# Bloque 2: validez y consistencia (antes: columnas de texto; después: columnas ya tipadas)
def validez_antes(df):
    return {
        "Fechas no estandarizadas":        df.filter(~F.col("fecha").rlike(r"^\d{4}-\d{2}-\d{2}$")).count(),
        "Precios no numéricos":            df.filter(~F.col("precio_unitario").rlike(r"^\d+$")).count(),
        "Cantidades no válidas (<= 0)":    df.filter(F.col("cantidad").cast("int") <= 0).count(),
        "Variantes de nombre de producto": df.select("producto_nombre").distinct().count(),
    }

def validez_despues(df):
    return {
        "Fechas no estandarizadas":        df.filter(F.col("fecha").isNull()).count(),
        "Precios no numéricos":            df.filter(F.col("precio_unitario").isNull()).count(),
        "Cantidades no válidas (<= 0)":    df.filter(F.col("cantidad") <= 0).count(),
        "Variantes de nombre de producto": df.select("producto_nombre").distinct().count(),
    }

antes   = {**perfil_antes,   **validez_antes(df_ventas_unificado)}
despues = {**perfil_despues, **validez_despues(df_ventas)}

filas_informe = [(k, int(antes[k]), int(despues[k]), int(despues[k]) - int(antes[k])) for k in antes]
informe_calidad = spark.createDataFrame(filas_informe, ["indicador", "antes", "despues", "variacion"])
print("INFORME DE CALIDAD: ANTES / DESPUÉS")
informe_calidad.show(truncate=False)

# Registros marcados por las reglas de validación (se conservan en df_ventas)
print("REGISTROS MARCADOS (no eliminados)")
print(f"  Regla R1 violada (precio_total < 0):      {df_ventas.filter(~F.col('regla_precio_total_ok')).count()}")
print(f"  Outliers IQR en precio_unitario:          {df_ventas.filter(F.col('es_outlier_iqr')).count()}")
print(f"  Regla R2 violada (precio_unitario < 1000): {df_ventas.filter(F.col('precio_bajo_sospechoso')).count()}")

# Bitácora de trazabilidad: una entrada por paso (último conteo), en orden de ocurrencia
pasos = {}
for paso, n in bitacora:
    pasos[paso] = n
print("\nBITÁCORA DE TRAZABILIDAD")
spark.createDataFrame(list(pasos.items()), ["paso", "filas"]).show(truncate=False)

INFORME DE CALIDAD: ANTES / DESPUÉS
+-------------------------------+-----+-------+---------+
|indicador                      |antes|despues|variacion|
+-------------------------------+-----+-------+---------+
|Filas totales                  |315  |274    |-41      |
|Registros duplicados exactos   |15   |0      |-15      |
|Nulos en id_transaccion        |0    |0      |0        |
|Nulos en fecha                 |0    |0      |0        |
|Nulos en id_producto           |0    |0      |0        |
|Nulos en producto_nombre       |0    |0      |0        |
|Nulos en categoria             |46   |0      |-46      |
|Nulos en cantidad              |0    |0      |0        |
|Nulos en precio_unitario       |0    |0      |0        |
|Nulos en id_cliente            |0    |0      |0        |
|Nulos en sucursal              |0    |0      |0        |
|Fechas no estandarizadas       |226  |0      |-226     |
|Precios no numéricos           |203  |0      |-203     |
|Cantidades no válidas (<= 0)   |27 

%md
**Resultado.** El informe comparativo muestra que el proceso de limpieza resolvió la totalidad de los problemas de calidad detectados:

| Dimensión | Indicador | Antes | Después |
|---|---|---|---|
| Volumen | Filas totales | $315$ | $274$ |
| Unicidad | Registros duplicados exactos | $15$ | $0$ |
| Completitud | Nulos en `categoria` | $46$ | $0$ |
| Completitud | Nulos en las 8 columnas restantes | $0$ | $0$ |
| Validez | Fechas no estandarizadas | $226$ | $0$ |
| Validez | Precios no numéricos | $203$ | $0$ |
| Validez | Cantidades no válidas ($\le 0$) | $27$ | $0$ |
| Consistencia | Variantes de nombre de producto | $31$ | $10$ |

Antes de la limpieza, el $\frac{226}{315} \approx 71{,}7\%$ de las fechas y el $\frac{203}{315} \approx 64{,}4\%$ de los precios no podían usarse directamente en cálculos. Después, el $100\%$ de ambas columnas tiene un tipo de dato estándar. Las cantidades no válidas figuran como $27$ en el conjunto inicial y no como las $26$ eliminadas en el paso 3.1, porque una de ellas (TRX10053, cantidad $0$) estaba duplicada y se contabilizó al eliminar duplicados.

**Registros marcados.** La regla $R_1$ no registra violaciones. Se conservan marcados $7$ outliers por IQR y $7$ precios bajo $\$1.000$ (regla $R_2$), que en conjunto suman $14$ registros ($\approx 5{,}1\%$ de `df_ventas`).

**Bitácora de trazabilidad.** El recorrido de las filas queda completamente explicado:

$$
\underbrace{105 + 125 + 85}_{\text{carga}} = \underbrace{315}_{\text{unión}}
\;\xrightarrow{\;-15 \text{ duplicados}\;}\; 300
\;\xrightarrow{\;-26 \text{ cantidad} \le 0\;}\; 274
$$

La diferencia total de $315 - 274 = 41$ filas se descompone íntegramente en $15$ duplicados y $26$ cantidades no válidas, sin pérdidas sin justificar. El conjunto final conserva el $\frac{274}{315} \approx 87{,}0\%$ de los registros originales, y todos los descartes tienen una causa documentada.

%md
## Parte 5. Análisis con Spark SQL (ID3.3)

### 5.1 Creación de la vista temporal `ventas_limpias`

Spark SQL permite consultar un DataFrame con lenguaje SQL estándar una vez registrado como **vista temporal**. La vista no copia los datos: es un nombre que apunta al plan de ejecución del DataFrame y existe solo durante la sesión de Spark. Se crea con `createOrReplaceTempView()`, que la reemplaza si ya existía; así el notebook puede reejecutarse sin errores.

La vista se crea a partir de `df_ventas`, el DataFrame final limpio y validado ($274$ filas). Incluye las nueve columnas originales ya tipadas, las columnas calculadas `precio_total` y `dia_semana`, y las marcas de calidad `es_outlier_iqr` y `precio_bajo_sospechoso`, que permiten a las consultas distinguir los registros atípicos.

Para comprobar que la vista quedó registrada correctamente, se describe su estructura y se verifica que el número de filas consultado con SQL coincide con el del DataFrame.

**Decisión técnica (presentación de resultados).** Las consultas se ejecutan con `spark.sql()`. Los montos se convierten a `BIGINT`, ya que Spark muestra los valores `double` grandes en notación científica (por ejemplo, `7.9999992E7`), lo que dificulta su lectura.

In [0]:
# Registro del DataFrame final como vista temporal SQL
df_ventas.createOrReplaceTempView("ventas_limpias")

# Estructura de la vista
print("Estructura de la vista ventas_limpias:")
spark.sql("DESCRIBE ventas_limpias").show(truncate=False)

# Control: el conteo vía SQL debe coincidir con el del DataFrame
n_sql = spark.sql("SELECT COUNT(*) AS filas FROM ventas_limpias").first()["filas"]
print(f"Filas en ventas_limpias (SQL): {n_sql} | Filas en df_ventas: {df_ventas.count()}")
assert n_sql == df_ventas.count()

Estructura de la vista ventas_limpias:
+----------------------+---------+-------+
|col_name              |data_type|comment|
+----------------------+---------+-------+
|id_transaccion        |string   |NULL   |
|fecha                 |date     |NULL   |
|id_producto           |string   |NULL   |
|producto_nombre       |string   |NULL   |
|categoria             |string   |NULL   |
|cantidad              |int      |NULL   |
|precio_unitario       |double   |NULL   |
|id_cliente            |string   |NULL   |
|sucursal              |string   |NULL   |
|precio_total          |double   |NULL   |
|dia_semana_num        |int      |NULL   |
|dia_semana            |string   |NULL   |
|regla_precio_total_ok |boolean  |NULL   |
|es_outlier_iqr        |boolean  |NULL   |
|precio_bajo_sospechoso|boolean  |NULL   |
+----------------------+---------+-------+

Filas en ventas_limpias (SQL): 274 | Filas en df_ventas: 274


%md
**Resultado.** La vista `ventas_limpias` quedó registrada con $15$ columnas, cada una con su tipo de dato definitivo:

| Grupo | Columnas | Tipos |
|---|---|---|
| Originales (tipadas) | `id_transaccion`, `fecha`, `id_producto`, `producto_nombre`, `categoria`, `cantidad`, `precio_unitario`, `id_cliente`, `sucursal` | `string`, `date`, `int`, `double` |
| Calculadas | `precio_total`, `dia_semana`, `dia_semana_num` | `double`, `string`, `int` |
| Marcas de calidad | `regla_precio_total_ok`, `es_outlier_iqr`, `precio_bajo_sospechoso` | `boolean` |

El conteo realizado con SQL ($274$ filas) coincide con el del DataFrame (`assert` superado), lo que confirma que la vista refleja fielmente el conjunto limpio. Que `fecha` sea `date` y `precio_unitario` y `precio_total` sean `double` permite a las consultas ordenar cronológicamente y agregar montos sin conversiones adicionales.

%md
### 5.2 Consulta 1: las 10 ventas con mayor `precio_total`

**Pregunta de negocio:** ¿cuáles son las transacciones de mayor monto del trimestre?

La consulta selecciona `id_transaccion`, `fecha`, `sucursal`, `producto_nombre` y `precio_total`, ordena de mayor a menor y limita el resultado a $10$ filas. Se agrega `id_transaccion` como segundo criterio de orden para que los empates en monto se resuelvan siempre del mismo modo y el resultado sea reproducible.

**Consulta complementaria.** Como la Parte 4 identificó precios unitarios atípicos que representan el $60{,}3\%$ del monto total, se ejecuta la misma consulta excluyendo los registros marcados (`es_outlier_iqr` y `precio_bajo_sospechoso`). La comparación muestra cómo cambia la respuesta a la pregunta de negocio según se consideren o no esos registros.

In [0]:
# Consulta 1: top 10 por precio_total (columnas exigidas por el enunciado)
print("Consulta 1: 10 ventas con mayor precio_total")
spark.sql("""
    SELECT id_transaccion,
           fecha,
           sucursal,
           producto_nombre,
           CAST(precio_total AS BIGINT) AS precio_total
    FROM ventas_limpias
    ORDER BY precio_total DESC, id_transaccion
    LIMIT 10
""").show(truncate=False)

# Complementaria: misma consulta excluyendo los registros marcados en la Parte 4
print("Consulta 1 complementaria: excluyendo outliers IQR y precios bajo $1.000")
spark.sql("""
    SELECT id_transaccion,
           fecha,
           sucursal,
           producto_nombre,
           CAST(precio_total AS BIGINT) AS precio_total
    FROM ventas_limpias
    WHERE NOT es_outlier_iqr
      AND NOT precio_bajo_sospechoso
    ORDER BY precio_total DESC, id_transaccion
    LIMIT 10
""").show(truncate=False)

Consulta 1: 10 ventas con mayor precio_total
+--------------+----------+------------+---------------------+------------+
|id_transaccion|fecha     |sucursal    |producto_nombre      |precio_total|
+--------------+----------+------------+---------------------+------------+
|TRX20047      |2024-12-07|Santiago    |router wifi          |79999992    |
|TRX30045      |2024-10-01|Concepción  |laptop gamer         |79999920    |
|TRX20001      |2024-11-23|Santiago    |disco ssd 1tb        |69999930    |
|TRX20119      |2024-12-19|Santiago    |auriculares bluetooth|69999930    |
|TRX10037      |2024-11-10|Viña del Mar|webcam hd            |35555552    |
|TRX20037      |2024-12-16|Santiago    |mouse inalambrico    |29999970    |
|TRX30009      |2024-10-20|Concepción  |impresora laser      |19999980    |
|TRX10074      |2024-11-25|Viña del Mar|teclado mecanico     |4499900     |
|TRX20014      |2024-10-15|Santiago    |teclado mecanico     |4499900     |
|TRX20063      |2024-10-17|Santiago    |lap

%md
**Resultado e interpretación.**

**Consulta 1 (todos los registros).** Las siete primeras posiciones corresponden exactamente a los $7$ outliers identificados en el paso 4.2, con montos entre $\$19.999.980$ y $\$79.999.992$. Recién en la octava posición aparece una venta con precio regular ($\$4.499.900$). Santiago concentra $4$ de esos $7$ registros atípicos, entre ellos el primero del ranking (un *router wifi* por $\$79.999.992$). Tomada sin filtrar, la consulta llevaría a conclusiones erróneas: atribuiría las mayores ventas del trimestre a productos de bajo valor, como un mouse o unos auriculares, solo porque sus precios unitarios son valores de relleno.

**Consulta complementaria (sin registros marcados).** Excluidos los $14$ registros marcados, la venta de mayor monto es de $\$4.499.900$, correspondiente a $10$ unidades de *teclado mecanico* a $\$449.990$ (empate entre Viña del Mar y Santiago). El monto máximo real es así unas $\frac{79.999.992}{4.499.900} \approx 17{,}8$ veces menor que el aparente. Las diez ventas válidas de mayor monto se reparten entre Santiago ($4$), Viña del Mar ($4$) y Concepción ($2$), y todas combinan un precio unitario alto ($\$349.990$ a $\$449.990$) con cantidades de $7$ a $10$ unidades.

**Hallazgo.** La comparación cuantifica el valor práctico del control de calidad de la Parte 4: sin las marcas de outliers, el ranking de ventas más importantes del trimestre estaría formado en un $70\%$ ($7$ de $10$) por datos erróneos. Las marcas permiten obtener la respuesta correcta sin eliminar información del conjunto.

%md
### 5.3 Consulta 2: total de ventas y cantidad de transacciones por sucursal

**Pregunta de negocio:** ¿cuánto vendió cada sucursal en el trimestre y con cuántas transacciones?

La consulta agrupa por `sucursal` y calcula $\sum \text{precio\_total}$ y el número de transacciones (`COUNT(*)`), ordenando de mayor a menor venta. Para enriquecer la lectura se agregan dos indicadores derivados:

$$
\text{participación}_s = 100 \cdot \frac{\sum_{i \in s} \text{precio\_total}_i}{\sum_{i} \text{precio\_total}_i},
\qquad
\text{ticket promedio}_s = \frac{\sum_{i \in s} \text{precio\_total}_i}{n_s}
$$

La participación se calcula con la función de ventana `SUM(SUM(precio_total)) OVER ()`, que obtiene el total general sin necesidad de una subconsulta.

**Consulta complementaria.** Al igual que en la Consulta 1, se repite el cálculo excluyendo los registros marcados en la Parte 4, para evaluar si los outliers alteran la jerarquía entre sucursales.

In [0]:
consulta_2 = """
    SELECT sucursal,
           COUNT(*)                                                           AS transacciones,
           CAST(SUM(precio_total) AS BIGINT)                                  AS total_ventas,
           ROUND(100 * SUM(precio_total) / SUM(SUM(precio_total)) OVER (), 1) AS pct_ventas,
           CAST(ROUND(AVG(precio_total)) AS BIGINT)                           AS ticket_promedio
    FROM ventas_limpias
    {filtro}
    GROUP BY sucursal
    ORDER BY total_ventas DESC
"""

# Consulta 2: todos los registros
print("Consulta 2: total de ventas y transacciones por sucursal")
spark.sql(consulta_2.format(filtro="")).show(truncate=False)

# Complementaria: excluyendo los registros marcados en la Parte 4
print("Consulta 2 complementaria: excluyendo outliers IQR y precios bajo $1.000")
spark.sql(consulta_2.format(filtro="WHERE NOT es_outlier_iqr AND NOT precio_bajo_sospechoso")).show(truncate=False)

Consulta 2: total de ventas y transacciones por sucursal
+------------+-------------+------------+----------+---------------+
|sucursal    |transacciones|total_ventas|pct_ventas|ticket_promedio|
+------------+-------------+------------+----------+---------------+
|Santiago    |112          |358320486   |56.1      |3199290        |
|Concepción  |70           |156110410   |24.4      |2230149        |
|Viña del Mar|92           |124594447   |19.5      |1354287        |
+------------+-------------+------------+----------+---------------+

Consulta 2 complementaria: excluyendo outliers IQR y precios bajo $1.000
+------------+-------------+------------+----------+---------------+
|sucursal    |transacciones|total_ventas|pct_ventas|ticket_promedio|
+------------+-------------+------------+----------+---------------+
|Santiago    |105          |108320510   |42.7      |1031624        |
|Viña del Mar|89           |89038880    |35.1      |1000437        |
|Concepción  |66           |56110260    |

%md
**Resultado e interpretación.**

| Sucursal | Transacciones | Ventas (todos) | Part. (todos) | Ventas (sin marcados) | Part. (sin marcados) | Ticket prom. (sin marcados) |
|---|---|---|---|---|---|---|
| Santiago | $112$ | $\$358.320.486$ | $56{,}1\%$ | $\$108.320.510$ | $42{,}7\%$ | $\$1.031.624$ |
| Concepción | $70$ | $\$156.110.410$ | $24{,}4\%$ | $\$56.110.260$ | $22{,}1\%$ | $\$850.155$ |
| Viña del Mar | $92$ | $\$124.594.447$ | $19{,}5\%$ | $\$89.038.880$ | $35{,}1\%$ | $\$1.000.437$ |

**Consulta 2 (todos los registros).** Santiago lidera con el $56{,}1\%$ de las ventas, seguida de Concepción ($24{,}4\%$) y Viña del Mar ($19{,}5\%$). Esta jerarquía es, sin embargo, un artefacto de los outliers: en Santiago, los $4$ registros atípicos aportan cerca de $\$250$ millones (alrededor del $70\%$ de su total), y en Concepción, los $2$ atípicos suman $\$99.999.900$ ($\approx 64\%$ del suyo).

**Consulta complementaria (sin registros marcados).** Al excluir los $14$ registros marcados, **el orden entre sucursales cambia**: Viña del Mar pasa del tercer al segundo lugar ($35{,}1\%$) y Concepción baja al tercero ($22{,}1\%$). Santiago conserva el liderazgo ($42{,}7\%$), y su participación en ventas resulta muy cercana a su participación en transacciones ($\frac{105}{260} \approx 40{,}4\%$). Esto indica que su liderazgo se explica por un **mayor volumen de operaciones** y no por ventas de mayor valor. Los tickets promedio son similares entre sucursales (entre $\$0{,}85$ y $\$1{,}03$ millones); Concepción presenta el ticket más bajo y el menor número de transacciones válidas.

**Hallazgo.** La decisión de negocio cambia según se consideren o no los registros marcados. Con los datos sin depurar se concluiría que Concepción es la segunda sucursal en ventas, cuando en realidad es la de menor desempeño. Este resultado ilustra concretamente por qué el control de calidad es una condición previa para la toma de decisiones.

%md
### 5.4 Consulta 3: precio unitario promedio, mínimo y máximo por categoría

**Pregunta de negocio:** ¿cómo se distribuyen los precios de los productos en cada categoría?

La consulta agrupa por `categoria` y calcula `AVG`, `MIN` y `MAX` de `precio_unitario`, junto con el número de transacciones de cada grupo, ordenando por precio promedio descendente. Como medida de dispersión de cada categoría se puede leer el rango $\max - \min$.

**Consultas complementarias.**

1. La misma consulta excluyendo los registros marcados, para obtener estadísticos no distorsionados por los valores de relleno ($\$9.999.990$, $\$8.888.888$) ni por los precios implausibles ($\$1$, $\$50$).
2. El rango de precios y el número de precios distintos **por producto**. El paso 2.1 mostró que la categoría no es un atributo consistente del producto; esta consulta permite evaluar si el precio sí lo es. Si cada producto tuviera un precio propio, cabría esperar un solo valor o un rango estrecho por producto.

In [0]:
consulta_3 = """
    SELECT categoria,
           COUNT(*)                                    AS transacciones,
           CAST(ROUND(AVG(precio_unitario)) AS BIGINT) AS precio_promedio,
           CAST(MIN(precio_unitario) AS BIGINT)        AS precio_minimo,
           CAST(MAX(precio_unitario) AS BIGINT)        AS precio_maximo
    FROM ventas_limpias
    {filtro}
    GROUP BY categoria
    ORDER BY precio_promedio DESC
"""
filtro_marcados = "WHERE NOT es_outlier_iqr AND NOT precio_bajo_sospechoso"

# Consulta 3: todos los registros
print("Consulta 3: precio unitario promedio, mínimo y máximo por categoría")
spark.sql(consulta_3.format(filtro="")).show(truncate=False)

# Complementaria 1: excluyendo los registros marcados en la Parte 4
print("Consulta 3 complementaria: excluyendo outliers IQR y precios bajo $1.000")
spark.sql(consulta_3.format(filtro=filtro_marcados)).show(truncate=False)

# Complementaria 2: consistencia del precio por producto (sin registros marcados)
print("Rango de precio unitario por producto (sin registros marcados)")
spark.sql(f"""
    SELECT producto_nombre,
           CAST(MIN(precio_unitario) AS BIGINT) AS precio_minimo,
           CAST(MAX(precio_unitario) AS BIGINT) AS precio_maximo,
           COUNT(DISTINCT precio_unitario)      AS precios_distintos
    FROM ventas_limpias
    {filtro_marcados}
    GROUP BY producto_nombre
    ORDER BY producto_nombre
""").show(truncate=False)

Consulta 3: precio unitario promedio, mínimo y máximo por categoría
+--------------+-------------+---------------+-------------+-------------+
|categoria     |transacciones|precio_promedio|precio_minimo|precio_maximo|
+--------------+-------------+---------------+-------------+-------------+
|Sin categoría |40           |1148464        |50           |9999990      |
|Audio         |50           |549928         |1            |9999990      |
|Redes         |42           |392706         |1            |9999990      |
|Almacenamiento|49           |200602         |25990        |449990       |
|Periféricos   |45           |177102         |50           |449990       |
|Computación   |48           |144365         |1            |449990       |
+--------------+-------------+---------------+-------------+-------------+

Consulta 3 complementaria: excluyendo outliers IQR y precios bajo $1.000
+--------------+-------------+---------------+-------------+-------------+
|categoria     |transacciones|pre

%md
**Resultado e interpretación.**

**Consulta 3 (todos los registros).** Los promedios están fuertemente distorsionados por los valores atípicos. "Sin categoría" ($\$1.148.464$), Audio ($\$549.928$) y Redes ($\$392.706$) encabezan el ranking solo porque concentran precios de relleno: su máximo es $\$9.999.990$, y sus mínimos de $\$1$ o $\$50$ también son implausibles. Las otras tres categorías, sin outliers altos, tienen un máximo de $\$449.990$.

**Complementaria 1 (sin registros marcados).** Al excluir los $14$ registros marcados, los promedios se ubican en un rango estrecho, entre $\$147.437$ (Computación) y $\$201.419$ (Sin categoría), y las $6$ categorías comparten **exactamente** el mismo mínimo ($\$25.990$) y el mismo máximo ($\$449.990$). Audio, por ejemplo, pasa de un promedio aparente de $\$549.928$ a uno real de $\$187.120$, una reducción del $\frac{549.928 - 187.120}{549.928} \approx 66\%$.

**Complementaria 2 (precio por producto).** Los $10$ productos recorren prácticamente el mismo rango de precios ($\$25.990$ a $\$449.990$; solo *monitor 27 pulgadas* parte en $\$45.990$), con entre $8$ y $10$ precios distintos cada uno. Un mouse inalámbrico y un laptop gamer se venden, por tanto, al mismo conjunto de precios.

**Hallazgo.** El precio unitario no depende ni de la categoría ni del producto: los registros toman sus precios de un mismo conjunto de valores, con independencia de lo que se vende. Esto, sumado a la asignación incoherente de categorías observada en el paso 2.1, indica que **los atributos `categoria` y `precio_unitario` no son confiables en los sistemas de origen**, aun después de la limpieza. En consecuencia:

1. las diferencias de precio entre categorías que muestra la Consulta 3 depurada son pequeñas y no deben interpretarse como diferencias reales de posicionamiento comercial;
2. la limpieza técnica resuelve los problemas de formato, completitud y unicidad, pero no puede corregir un dato que es incorrecto en origen. Para un análisis comercial por categoría o producto sería necesario contrastar las ventas con un **maestro de productos** que defina la categoría y el precio de lista de cada artículo.

%md
## Parte 6. Conclusiones

### 6.1 Síntesis del flujo

Se implementó en Databricks un flujo completo de ingeniería de datos sobre las ventas del cuarto trimestre de 2024 de tres sucursales. La trazabilidad del volumen de registros quedó documentada en cada etapa:

$$
\underbrace{105 + 125 + 85}_{\text{carga}} = 315
\;\xrightarrow{\;-15 \text{ duplicados}\;}\; 300
\;\xrightarrow{\;-26 \text{ cantidad} \le 0\;}\; 274 \text{ ventas válidas}
$$

El conjunto final conserva el $87{,}0\%$ de los registros originales, y cada descarte tiene una causa identificada y verificada mediante `assert`.

| Parte | Resultado principal |
|---|---|
| 1. Carga e integración | Tres CSV leídos como texto y unidos con `unionByName`; integridad verificada ($315 = 105 + 125 + 85$) |
| 2. Limpieza | $46$ nulos imputados, $15$ duplicados eliminados, $31 \rightarrow 10$ nombres de producto, $4$ formatos de fecha unificados, $192$ precios con símbolo convertidos a número |
| 3. Transformación | $26$ cantidades no válidas filtradas; `precio_total` y `dia_semana` creadas y verificadas; $3$ sucursales, $6$ categorías |
| 4. Validación | Regla $R_1$ cumplida al $100\%$; $7$ outliers IQR y $7$ precios bajo $\$1.000$ marcados; informe antes/después sin problemas residuales |
| 5. Spark SQL | Vista `ventas_limpias` y tres consultas, cada una con su contraste excluyendo registros marcados |

### 6.2 Decisiones técnicas

| Decisión | Justificación | Alternativa descartada |
|---|---|---|
| *Volume* de Unity Catalog en lugar del DBFS raíz | El DBFS raíz está deshabilitado en Databricks Free Edition | — |
| Lectura con todas las columnas como texto | Evita que Spark convierta en nulo un valor mal formateado sin dejar rastro | `inferSchema=True` |
| `unionByName` | Combina por nombre de columna, inmune a diferencias de orden | `union` (por posición) |
| `fillna("Sin categoría")` | Ningún producto tiene una categoría única; conserva el $14{,}6\%$ de las ventas sin inventar datos | `dropna` o imputación por producto |
| Diccionario de sinónimos en `producto_nombre` | Las minúsculas por sí solas dejaban $17$ variantes | Solo `lower` |
| Formato de fecha asignado por patrón (`when` + `rlike`) | En modo ANSI, `to_date` falla ante formatos no coincidentes | `coalesce` de intentos sucesivos |
| Filtrar (no corregir) cantidades $\le 0$ | No hay evidencia para reconstruir el valor correcto | Cambiar el signo |
| Marcar (no eliminar) outliers | La exclusión es una decisión de análisis, no de limpieza | Eliminar con `filter` |
| Regla complementaria $\text{precio\_unitario} \ge 1.000$ | Con $L_{\text{inf}} < 0$, el IQR no detecta precios anormalmente bajos | Solo IQR |

### 6.3 Hallazgos principales

1. **Los outliers dominan los montos.** $7$ registros ($2{,}6\%$) concentran el $60{,}3\%$ de las ventas. Sin su marca, $7$ de las $10$ mayores ventas del trimestre serían datos erróneos.
2. **La jerarquía de sucursales depende de la calidad de los datos.** Con todos los registros, Concepción aparece como segunda sucursal ($24{,}4\%$); excluidos los marcados, es la de menor desempeño ($22{,}1\%$), y Viña del Mar sube al segundo lugar ($35{,}1\%$). Santiago lidera en ambos escenarios por volumen de transacciones, no por mayor valor de venta.
3. **Categoría y precio no son confiables en origen.** Cada producto aparece en $3$ a $5$ categorías y en $8$ a $10$ precios distintos, todos dentro del mismo rango ($\$25.990$ a $\$449.990$). Las diferencias de precio entre categorías no reflejan un posicionamiento comercial real.

### 6.4 Limitaciones y recomendaciones

- La limpieza técnica corrige formato, completitud y unicidad, pero **no puede corregir un dato incorrecto en origen**. Se recomienda incorporar un **maestro de productos** con la categoría y el precio de lista de cada artículo, y validar las transacciones contra él.
- Los registros marcados se conservan; su tratamiento definitivo (exclusión, corrección o consulta al área responsable) debe acordarse con el negocio.
- Las cantidades negativas podrían corresponder a devoluciones. Si es así, deberían registrarse en una tabla separada en lugar de mezclarse con las ventas.
- El análisis abarca un solo trimestre ($274$ ventas válidas); patrones como la mayor actividad de los sábados deben confirmarse con un período más extenso.